In [1]:
import pandas as pd

# Leggiamo il file Excel. 
# index_col=0 dice a pandas di usare la prima colonna (le date) come indice delle righe
df = pd.read_excel('Data2026SP500.xlsx', index_col=0, header=0, parse_dates=True)
df_clean = df.dropna(axis=1)  # Rimuove le colonne con valori NaN

print(df_clean.shape[1])  # Stampa il numero di aziende rimaste dopo la pulizia


388


In [2]:
returns = ((df_clean - df_clean.shift(1)) / df_clean.shift(1)) * 100
returns = returns.dropna()
#matrice [T x 388 asset]
media_campionaria=returns.mean()
covarianza_campionaria=returns.cov()
print(media_campionaria.head())
print(covarianza_campionaria.head())

AMAZON.COM - TOT RETURN IND                 0.113075
ABBOTT LABORATORIES - TOT RETURN IND        0.048871
AES - TOT RETURN IND                        0.034396
INTERNATIONAL BUS.MCHS. - TOT RETURN IND    0.043124
ADVANCED MICRO DEVICES - TOT RETURN IND     0.120370
dtype: float64
                                          AMAZON.COM - TOT RETURN IND  \
AMAZON.COM - TOT RETURN IND                                  5.453952   
ABBOTT LABORATORIES - TOT RETURN IND                         0.883836   
AES - TOT RETURN IND                                         1.433967   
INTERNATIONAL BUS.MCHS. - TOT RETURN IND                     1.174818   
ADVANCED MICRO DEVICES - TOT RETURN IND                      2.954698   

                                          ABBOTT LABORATORIES - TOT RETURN IND  \
AMAZON.COM - TOT RETURN IND                                           0.883836   
ABBOTT LABORATORIES - TOT RETURN IND                                  1.888988   
AES - TOT RETURN IND               

In [3]:
import numpy as np
# La funzione eigh è ottimizzata per matrici simmetriche e reali
autovalori, autovettori = np.linalg.eigh(covarianza_campionaria)

# eigh restituisce gli autovalori in ordine crescente. Li invertiamo per avere i fattori principali all'inizio.
autovalori = autovalori[::-1]
autovettori = autovettori[:, ::-1]

for i in range(5):
    sum_autovalori = np.sum(autovalori[:i+1])
print(f"Somma dei primi {i+1} autovalori: {sum_autovalori}")

Somma dei primi 5 autovalori: 843.5014632806229


In [7]:
# 1. Somma totale di tutti i 388 autovalori (Varianza Totale del mercato)
varianza_totale = np.sum(autovalori)

# 2. Percentuale di varianza spiegata da ciascun singolo fattore
varianza_spiegata = (autovalori / varianza_totale) * 100

# 3. Somma cumulativa delle percentuali
varianza_cumulata = np.cumsum(varianza_spiegata)

# Stampiamo i risultati per i primi 10 fattori
for i in range(20):
    print(f"Fattore {i+1}: spiega il {varianza_spiegata[i]:.2f}% -> Cumulato: {varianza_cumulata[i]:.2f}%")

Fattore 1: spiega il 38.03% -> Cumulato: 38.03%
Fattore 2: spiega il 3.48% -> Cumulato: 41.51%
Fattore 3: spiega il 2.82% -> Cumulato: 44.33%
Fattore 4: spiega il 2.45% -> Cumulato: 46.78%
Fattore 5: spiega il 1.49% -> Cumulato: 48.27%
Fattore 6: spiega il 1.34% -> Cumulato: 49.61%
Fattore 7: spiega il 1.13% -> Cumulato: 50.74%
Fattore 8: spiega il 1.03% -> Cumulato: 51.77%
Fattore 9: spiega il 0.88% -> Cumulato: 52.65%
Fattore 10: spiega il 0.80% -> Cumulato: 53.45%
Fattore 11: spiega il 0.75% -> Cumulato: 54.20%
Fattore 12: spiega il 0.71% -> Cumulato: 54.91%
Fattore 13: spiega il 0.68% -> Cumulato: 55.59%
Fattore 14: spiega il 0.59% -> Cumulato: 56.18%
Fattore 15: spiega il 0.56% -> Cumulato: 56.73%
Fattore 16: spiega il 0.54% -> Cumulato: 57.27%
Fattore 17: spiega il 0.52% -> Cumulato: 57.79%
Fattore 18: spiega il 0.50% -> Cumulato: 58.30%
Fattore 19: spiega il 0.48% -> Cumulato: 58.78%
Fattore 20: spiega il 0.48% -> Cumulato: 59.26%


In [ ]:
#$R_t = \alpha + B F_t + \epsilon_t$, la matrice delle esposizioni $B$ (i famosi Betas) è esattamente la matrice composta dai 20 autovettori che hai appena selezionato
#$F_t = (R_t - \mu) \cdot \Gamma_K$
#dobbiamo calcolare la vera e propria serie storica di questi 20 fattori

# 1. Selezioniamo solo i primi 20 fattori (K=20)
K = 20
autovettori_K = autovettori[:, :K]  # Matrice [388 x 20] (Questa è la tua matrice B)
autovalori_K = autovalori[:K]       # Vettore di 20 elementi (Le varianze dei fattori)

# 2. Centriamo i rendimenti originali (sottraendo la media campionaria)
returns_centered = returns - media_campionaria

# 3. Calcoliamo la serie storica dei Fattori (Proiezione)
# Moltiplicazione matriciale: [T x 388] moltiplicato per [388 x 20] = [T x 20]
Ft = returns_centered.dot(autovettori_K)

# Diamo dei nomi alle nuove 20 colonne per comodità
Ft.columns = [f"Fattore_{i+1}" for i in range(K)]

print("Serie storica dei Fattori di Rischio:")
print(Ft.head())


Serie storica dei Fattori di Rischio:
               Fattore_1  Fattore_2  Fattore_3  Fattore_4  Fattore_5  \
Retrieving...                                                          
2006-01-02     -1.212849   0.219418  -0.228677   0.007322   0.041573   
2006-01-03     27.833079  -1.181244   1.117291  -9.471039   1.871150   
2006-01-04     12.031017  -5.670226   2.530319  -3.167498  -1.435990   
2006-01-05      2.634447  -1.060634   0.677954   7.363236   0.996856   
2006-01-06     20.661229  -6.578658   2.964118  -2.997002   1.196438   

               Fattore_6  Fattore_7  Fattore_8  Fattore_9  Fattore_10  \
Retrieving...                                                           
2006-01-02      0.019466   0.035261   0.024959  -0.011843    0.006031   
2006-01-03     -5.445402  -6.281564  -3.897689  -2.751874   -2.243960   
2006-01-04     -3.112419  -1.883107  -0.643647   2.483235    1.541617   
2006-01-05      0.111695  -4.870378  -1.328942   2.285162   -1.926106   
2006-01-06     -1.1